# Engine Test — plotsim

Standalone verification of the plotsim engine against `saas_engine_test.yaml` — a purpose-built template that isolates the core pipeline from known configuration-caused failures (lognorm shape distortion, aggressive correlations, partial archetype assignment).

**Question being answered:** Does the engine produce output where configured intent is observable when given correct input?

**Test template design choices (settled — not under discussion):**
- Beta over lognorm for mrr (linear position_to_center mapping)
- Conservative correlations ±0.40 / 0.35 (PD without Higham)
- Zero noise (clean signal)
- All 6 archetypes assigned to entities
- No metric_overrides
- Schema unchanged from sample_saas.yaml

**Phases:**
1. Generate and load
2. Shape recovery
3. Correlation sign match
4. Causal lag
5. Dimensional model

---
## Phase 1 — Generate and load

Verify:
- Engine runs without error or warning
- All 9 expected tables are produced
- Two same-seed runs produce byte-identical output (determinism contract)
- SCD Type 2 fires (or doesn't — and we identify why)

### Setup

Imports, plot style, constants. No external helper module — this notebook stands alone.

In [ ]:
"""Setup — imports, plot style, constants."""
import warnings
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "figure.figsize": (11, 5.5), "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "lines.linewidth": 1.5,
})

SEED = 42
CONFIG_PATH = Path.cwd().parent / "plotsim" / "configs" / "saas_engine_test.yaml"

COLORS = {
    "rocket_then_cliff": "#d62728",
    "steady_grower": "#2ca02c",
    "slow_death": "#8c564b",
    "seasonal_spiker": "#9467bd",
    "zombie_account": "#7f7f7f",
    "expansion_champion": "#1f77b4",
}

print("setup OK")

### Load config

Load `saas_engine_test.yaml` and print a structural summary. Watch for:
- All 6 archetypes should appear in both "declared" and "used"
- Noise should read `0.0` across all three channels
- If config load emits a Higham projection warning, the conservative correlations (±0.40, 0.35) didn't hold the matrix PD — note it

In [ ]:
"""Load config and print structural summary."""
from plotsim.config import load_config
from plotsim.tables import generate_tables_with_state

cfg = load_config(CONFIG_PATH)

print(f"config:              {CONFIG_PATH.name}")
print(f"seed:                {SEED}")
print(f"entities:            {[e.name for e in cfg.entities]}")
print(f"archetypes declared: {[a.name for a in cfg.archetypes]}")
print(f"archetypes used:     {sorted(set(e.archetype for e in cfg.entities))}")
print(f"metrics:             {[m.name for m in cfg.metrics]}")
print(f"correlation pairs:   {len(cfg.correlations)}")
print(f"tables declared:     {len(cfg.tables)}")
print(f"time window:         {cfg.time_window.start} -> {cfg.time_window.end}")
print(f"noise:               sigma={cfg.noise.gaussian_sigma}, "
      f"outlier={cfg.noise.outlier_rate}, MCAR={cfg.noise.mcar_rate}")

### Determinism

Two consecutive runs at the same seed must produce byte-identical CSV output and identical trajectory arrays. If this fails, every assertion downstream is meaningless.

In [ ]:
"""Generate tables twice at the same seed. Assert byte-identical output."""

def _csv_blob(tables_dict):
    """Concatenate every table's CSV bytes in sorted key order."""
    parts = []
    for name in sorted(tables_dict):
        buf = io.StringIO()
        tables_dict[name].to_csv(buf, index=False, lineterminator="\n")
        parts.append((name + "\n").encode("utf-8"))
        parts.append(buf.getvalue().encode("utf-8"))
    return b"".join(parts)

tables_a, state_a = generate_tables_with_state(cfg, np.random.default_rng(SEED))
tables_b, state_b = generate_tables_with_state(cfg, np.random.default_rng(SEED))

csv_a, csv_b = _csv_blob(tables_a), _csv_blob(tables_b)
byte_diff = sum(1 for x, y in zip(csv_a, csv_b) if x != y) + abs(len(csv_a) - len(csv_b))

assert byte_diff == 0, f"determinism breach: {byte_diff} byte differences"

for ent in cfg.entities:
    np.testing.assert_array_equal(
        state_a.trajectories[ent.name], state_b.trajectories[ent.name],
        err_msg=f"trajectory drift for {ent.name}",
    )

print(f"PASS — two same-seed runs identical: {len(csv_a):,} bytes")
print(f"trajectories: {len(state_a.trajectories)} entities, all bit-equal")

# Use run A going forward
tables, state = tables_a, state_a
n_periods = len(tables["dim_date"])
print(f"periods: {n_periods}")

### Output file inventory

All 9 tables from the schema should be present: 4 dims (`dim_date`, `dim_company`, `dim_user`, `dim_plan`), 3 facts (`fct_engagement`, `fct_revenue`, `fct_support_tickets`), 2 events (`evt_login`, `evt_churn`). Row counts serve as a sanity check — dims should be small, facts should be `n_entities × n_periods`, events are variable.

In [ ]:
"""Verify all expected tables were generated."""
expected_tables = [
    "dim_date", "dim_company", "dim_user", "dim_plan",
    "fct_engagement", "fct_revenue", "fct_support_tickets",
    "evt_login", "evt_churn",
]
present = [t for t in expected_tables if t in tables]
missing = [t for t in expected_tables if t not in tables]

print(f"present: {len(present)}/{len(expected_tables)}")
for t in present:
    df = tables[t]
    print(f"  {t:25s}  {df.shape[0]:6,} rows × {df.shape[1]} cols")
if missing:
    print(f"\nMISSING: {missing}")
else:
    print("\nPASS — all 9 tables generated")

### SCD Type 2 check

The SCD trigger is `fct_revenue.mrr` with thresholds `[0.4, 0.7]` and labels `[starter, growth, enterprise]`.

**Predicted issue:** mrr on this template ranges from 100 to 50,000. If the engine compares thresholds against realized metric values (not trajectory position), thresholds of 0.4 and 0.7 will never be crossed and SCD won't fire.

**If SCD doesn't fire:** Read `tables.py`, function `_compute_scd_versions` (lines 1703–1775) to confirm whether thresholds are trajectory-based or value-based. Report what the code does — don't adjust thresholds without understanding the code path.

In [ ]:
"""SCD Type 2 — check whether plan_tier transitions fired."""
from plotsim.config import parse_source, SCDType2Source

scd_col = next(
    (col for tbl in cfg.tables if tbl.name == "dim_company"
     for col in tbl.columns
     if isinstance(parse_source(col.source), SCDType2Source)),
    None,
)

if scd_col:
    print(f"SCD column:     dim_company.{scd_col.name}")
    print(f"trigger_metric: {scd_col.scd_type2.trigger_metric}")
    print(f"thresholds:     {scd_col.scd_type2.thresholds}")
    print(f"labels:         {scd_col.scd_type2.labels}")
else:
    print("No SCD column found in dim_company")

dim_company = tables["dim_company"]
n_entities = len(cfg.entities)
distinct_ids = dim_company["company_id"].nunique()
expanded_rows = len(dim_company)
scd_transitions = expanded_rows - distinct_ids

print(f"\ndistinct company_ids: {distinct_ids} (expected: {n_entities})")
print(f"total dim_company rows: {expanded_rows}")
print(f"SCD transitions: {scd_transitions}")

if scd_transitions == 0:
    print("\n⚠ SCD DID NOT FIRE")
    print("Thresholds [0.4, 0.7] likely compare against realized mrr ")
    print("values (100–50,000), not trajectory position (0–1).")
    print("ACTION: read tables.py _compute_scd_versions (lines 1703–1775).")
else:
    print(f"\nSCD fired {scd_transitions} transitions")
    dupes = dim_company[dim_company.duplicated("company_id", keep=False)]
    print(dupes.sort_values("company_id"))